In [2]:
# [Cell 1] (수정됨) 배치 사이즈 제한 및 모델 로드
import os
import glob
from tqdm import tqdm
import torch
import gc # 가비지 컬렉터

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_experimental.text_splitter import SemanticChunker

# === 경로 설정 ===
BASE_DIR = "."
DATA_DIR = os.path.join(BASE_DIR, "data", "processed", "06_text_merged")
DB_PATH = os.path.join(BASE_DIR, "data", "vector_store", "chroma_db")
MODEL_CACHE_DIR = os.path.join(BASE_DIR, "data", "model_cache")

os.makedirs(MODEL_CACHE_DIR, exist_ok=True)
os.makedirs(DB_PATH, exist_ok=True)

MODEL_NAME = "dragonkue/BGE-m3-ko" 

print(f"🚀 임베딩 모델 로드 중... (저장소: {MODEL_CACHE_DIR})")
device = "cuda" if torch.cuda.is_available() else "cpu"

# ★ 핵심 수정: batch_size를 8로 줄임 (VRAM 보호)
encode_kwargs = {
    'normalize_embeddings': True,
    'batch_size': 16 
}

embeddings = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
    model_kwargs={'device': device},
    encode_kwargs=encode_kwargs,
    cache_folder=MODEL_CACHE_DIR
)

# Semantic Chunker 설정
text_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile", 
    breakpoint_threshold_amount=95
)

print(f"✅ 모델 로드 완료 (Batch Size: 16)")

🚀 임베딩 모델 로드 중... (저장소: .\data\model_cache)
✅ 모델 로드 완료 (Batch Size: 16)


In [7]:
# [Cell 2] (수정됨) VRAM 방어형 청킹 로직

def split_large_text(text, max_chars=10000):
    """
    너무 긴 텍스트를 max_chars 단위로 안전하게 자름 (문단 기준)
    Semantic Chunker가 소화 불량에 걸리지 않게 '죽'을 만들어 주는 역할
    """
    chunks = []
    current_chunk = []
    current_length = 0
    
    # 문단(\n\n) 단위로 먼저 나눔
    paragraphs = text.split("\n\n")
    
    for para in paragraphs:
        # 문단 하나가 너무 길면 강제로 자름 (안전장치)
        if len(para) > max_chars:
            # 그냥 단순히 자르지만, 사실 이런 경우는 드뭄
            chunks.append(para) 
            continue
            
        if current_length + len(para) > max_chars:
            # 꽉 찼으면 저장하고 비움
            chunks.append("\n\n".join(current_chunk))
            current_chunk = [para]
            current_length = len(para)
        else:
            current_chunk.append(para)
            current_length += len(para) + 2 # \n\n 길이 포함
            
    if current_chunk:
        chunks.append("\n\n".join(current_chunk))
        
    return chunks

def load_and_chunk_files_safe(directory):
    files = sorted(glob.glob(os.path.join(directory, "*.txt")))
    all_splits = []
    
    print(f"📂 데이터 로드 및 청킹 시작... (대상: {len(files)}개)")
    print("   🛡️ VRAM 보호 모드: 파일을 나누어 처리합니다.")
    
    for file_path in tqdm(files):
        try:
            filename = os.path.basename(file_path)
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()
            
            if len(text) < 50: continue
            
            # 1. 전처리: 너무 긴 파일은 10000자 단위로 쪼갬 (Pre-chunking)
            text_blocks = split_large_text(text, max_chars=10000)
            
            file_docs = []
            for block in text_blocks:
                if not block.strip(): continue
                
                # 2. Semantic Chunking 수행 (작아진 블록 단위로 수행)
                splits = text_splitter.create_documents([block])
                file_docs.extend(splits)
            
            # 3. 메타데이터 주입
            for split in file_docs:
                split.metadata = {"source": filename}
                
            all_splits.extend(file_docs)
            
            # ★ 핵심: 파일 하나 끝나면 VRAM 청소
            del text_blocks, file_docs
            torch.cuda.empty_cache()
            gc.collect()
            
        except Exception as e:
            print(f"⚠️ 에러 발생 ({filename}): {e}")
            
    return all_splits

# 실행
splits = load_and_chunk_files_safe(DATA_DIR)

print(f"\n🎉 청킹 완료!")
print(f"   - 총 생성된 청크 수: {len(splits)}개")

# [Cell 2.5] 청크 데이터 저장 (검증용 + 재사용용)
import json
import os

# === 설정 ===
DEBUG_DIR = os.path.join(BASE_DIR, "data", "processed", "07_chunk_debug")
os.makedirs(DEBUG_DIR, exist_ok=True)

# 재사용을 위한 JSON 파일 경로
SAVE_FILE = os.path.join(DEBUG_DIR, "all_chunks_backup.json")

def save_chunks(splits):
    print(f"💾 청크 데이터 저장 중... (경로: {DEBUG_DIR})")
    
    # 1. [재사용용] JSON 저장 (나중에 load 가능)
    # Document 객체를 딕셔너리로 변환
    data_to_save = [
        {"page_content": doc.page_content, "metadata": doc.metadata} 
        for doc in splits
    ]
    
    with open(SAVE_FILE, "w", encoding="utf-8") as f:
        json.dump(data_to_save, f, ensure_ascii=False, indent=2)
        
    # 2. [검증용] 눈으로 보기 편한 텍스트 파일 생성
    # 파일별로 청크가 어떻게 나뉘었는지 보여줌
    debug_text_path = os.path.join(DEBUG_DIR, "visual_check.txt")
    
    with open(debug_text_path, "w", encoding="utf-8") as f:
        f.write(f"=== 총 {len(splits)}개의 청크가 생성됨 ===\n\n")
        
        for i, doc in enumerate(splits[:50]): # 상위 50개만 샘플로 저장 (전체는 너무 큼)
            source = doc.metadata.get('source', 'unknown')
            f.write(f"--- [Chunk {i+1}] (출처: {source}) ---\n")
            f.write(doc.page_content)
            f.write("\n\n" + "="*30 + "\n\n")

    print(f"✅ 저장 완료!")
    print(f"   - 재사용 파일: {SAVE_FILE} (나중에 이걸 로드하면 청킹 시간 단축!)")
    print(f"   - 눈으로 확인: {debug_text_path} (열어서 잘 잘렸나 보세요)")

# === 실행 ===
if 'splits' in locals() and splits:
    save_chunks(splits)
else:
    print("⚠️ 'splits' 변수가 없습니다. 셀 2를 먼저 실행하세요.")

# [Tip] 저장된 청크 불러오기 코드 (나중에 사용)
def load_chunks_from_json():
    import json
    from langchain_core.documents import Document
    
    file_path = os.path.join("data", "processed", "07_chunk_debug", "all_chunks_backup.json")
    
    if not os.path.exists(file_path):
        print("❌ 저장된 파일이 없습니다.")
        return []
        
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    # 다시 Document 객체로 변환
    loaded_splits = [
        Document(page_content=d["page_content"], metadata=d["metadata"]) 
        for d in data
    ]
    print(f"⚡ {len(loaded_splits)}개 청크 로드 완료!")
    return loaded_splits

# 사용법: splits = load_chunks_from_json()

📂 데이터 로드 및 청킹 시작... (대상: 100개)
   🛡️ VRAM 보호 모드: 파일을 나누어 처리합니다.


100%|██████████| 100/100 [19:15<00:00, 11.55s/it]



🎉 청킹 완료!
   - 총 생성된 청크 수: 4992개
💾 청크 데이터 저장 중... (경로: .\data\processed\07_chunk_debug)
✅ 저장 완료!
   - 재사용 파일: .\data\processed\07_chunk_debug\all_chunks_backup.json (나중에 이걸 로드하면 청킹 시간 단축!)
   - 눈으로 확인: .\data\processed\07_chunk_debug\visual_check.txt (열어서 잘 잘렸나 보세요)


In [4]:
# [Cell 2.9] 긴급 복구: 저장된 JSON에서 청크 불러오기
import json
import os
from langchain_core.documents import Document

# 경로 설정 (아까 저장한 그 경로)
BACKUP_FILE = os.path.join(".", "data", "processed", "07_chunk_debug", "all_chunks_backup.json")

def recover_splits():
    if not os.path.exists(BACKUP_FILE):
        print(f"❌ 복구 실패: 파일을 찾을 수 없습니다.\n경로: {BACKUP_FILE}")
        return []
    
    print(f"♻️ 데이터 복구 시작... (경로: {BACKUP_FILE})")
    
    with open(BACKUP_FILE, "r", encoding="utf-8") as f:
        data_loaded = json.load(f)
    
    # JSON 딕셔너리 -> Document 객체로 변환
    recovered_splits = []
    for item in data_loaded:
        doc = Document(
            page_content=item["page_content"],
            metadata=item["metadata"]
        )
        recovered_splits.append(doc)
        
    print(f"✅ 복구 완료! 총 {len(recovered_splits)}개의 청크가 메모리로 돌아왔습니다.")
    return recovered_splits

# 실행해서 splits 변수에 다시 할당
splits = recover_splits()

# 확인
if splits:
    print(f"   - 샘플 확인: {splits[0].page_content[:50]}...")

♻️ 데이터 복구 시작... (경로: .\data\processed\07_chunk_debug\all_chunks_backup.json)
✅ 복구 완료! 총 4992개의 청크가 메모리로 돌아왔습니다.
   - 샘플 확인: 2024년 ｢벤처확인종합관리시스템 기능 고도화｣ 용역사업 - (복수의결권주식, 스톡옵션, ...


In [ ]:
# [Cell 3] (수정됨) VRAM 방어 및 진행률 표시가 추가된 저장 로직
import gc
import torch
from tqdm import tqdm

# === 설정 ===
BATCH_SIZE = 32  # 16~32 정도가 안전함 (BGE-M3 모델이 큼)

print(f"💾 ChromaDB 초기화 및 저장 시작... (경로: {DB_PATH})")
print(f"   🛡️ 안전 모드: {BATCH_SIZE}개씩 나누어 저장합니다.")

# 1. ChromaDB 인스턴스 먼저 생성 (데이터 없이 껍데기만)
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory=DB_PATH,
    collection_name="government_proposals"
)

# 2. 배치 단위로 나누어 추가 (Batch Injection)
total_chunks = len(splits)

# tqdm으로 진행바 생성
for i in tqdm(range(0, total_chunks, BATCH_SIZE), desc="DB 저장 중"):
    # 배칠 슬라이싱
    batch = splits[i : i + BATCH_SIZE]
    
    try:
        # DB에 추가 (여기서 임베딩 계산이 일어남)
        vector_store.add_documents(batch)
        
        # ★ VRAM 청소 (매 배치마다 찌꺼기 제거)
        if device == "cuda":
            torch.cuda.empty_cache()
            
    except Exception as e:
        print(f"\n⚠️ 배치 저장 중 오류 발생 (Index {i}): {e}")
        # 오류 나도 멈추지 않고 다음 배치 시도 (선택 사항)
        continue

# 3. 강제 저장 (Persist) - 최신 버전 LangChain은 자동이지만 안전하게
# (버전에 따라 필요 없을 수 있지만 에러 안 나면 두는 게 좋음)
try:
    # 구버전 호환용 (신버전은 자동 저장됨)
    vector_store.persist()
except:
    pass

print(f"\n✅ 모든 저장 완료! (총 {total_chunks}개 청크)")
print(f"   📁 DB 경로: {DB_PATH}")

💾 ChromaDB 초기화 및 저장 시작... (경로: .\data\vector_store\chroma_db)
   🛡️ 안전 모드: 32개씩 나누어 저장합니다.


DB 저장 중: 100%|██████████| 156/156 [12:15<00:00,  4.71s/it]


✅ 모든 저장 완료! (총 4992개 청크)
   📁 DB 경로: .\data\vector_store\chroma_db


In [6]:
# [Cell 4] 검색 성능 테스트 (예산 확인)

# 네가 궁금해했던 바로 그 질문!
query = "벤처확인종합관리시스템 기능 고도화 용역사업의 예산은 얼마인가요?" 

print(f"🔍 질문: {query}")
print("-" * 50)

# 유사도 검색 (가장 관련성 높은 3개 문서 추출)
# k=3 : 상위 3개만 가져옴
results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"📄 [증거 자료 {i+1}] (출처: {doc.metadata.get('source', 'Unknown')})")
    print(f"내용:\n{doc.page_content[:500]}...") # 내용 길면 500자만 출력
    print("-" * 50)

🔍 질문: 벤처확인종합관리시스템 기능 고도화 용역사업의 예산은 얼마인가요?
--------------------------------------------------
📄 [증거 자료 1] (출처: (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .txt)
내용:
2024년 ｢벤처확인종합관리시스템 기능 고도화｣ 용역사업 - (복수의결권주식, 스톡옵션, 성과조건부주식) - 제안요청서 2024. 03. 목 차 1. 추진개요 · 3 2. 추진방안 · 5 3. 추진내용 · 9 4. 제안요청내용 · 24 5. 입찰관련사항 · 78 6. 제안서작성요령 · 82 7....
--------------------------------------------------
📄 [증거 자료 2] (출처: (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .txt)
내용:
|

Ⅶ. 별지서식 및 붙임 □ 별지서식 <별지서식 1호> 2024년 벤처확인종합관리시스템 기능 고도화 용역사업 제안서 표지 <별지서식 2호> 제안요청서 수용 여부 참조표 <별지서식 3호> 일반현황 및 연혁 <별지서식 4호> 자본금 및 매출액 현황 <별지서식 5호> 사업수행 조직도 <별지서식 6호> 기술적용계획표 <별지서식 7호> 소프트웨어사업 하도급 계획서 <별지서식 7-1호> 하도급 적정성 판단 자기평가표 <별지서식 8호> 공동수급표준협정서(공동이행방식) <별지서식 9호> 합의각서 □ 붙임자료
[붙임1] 청렴서약서
[붙임2] 근로자 권리보호 이행 서약서
[붙임3] 보안서약서
[붙임4] 정보화 용역사업 보안특약
[붙임5] 소프트웨어 개발사업의 적정 사업기간 종합 산정서
[붙임6] 소프트웨어 영향평가 검토결과서
- 94 -

(별지 제1호 서식) 제안서 표지 접수번호 2024년 벤처확인종합관리시스템 기능 고도화 용역사업 - 용역 제안서 - 사업자 기 관 명 등록번호 1)신청기관 대표...
-------------------------------------------------

In [7]:
# [Cell 4.5] 디버깅: 예산 데이터가 실제로 존재하는가?

# 1. 원본 청크에서 직접 찾아보기 (Keyword Search)
target_keyword = "352,000,000"
found_docs = []

print(f"🕵️‍♂️ '{target_keyword}' 데이터 수색 중...")

# 메모리에 있는 splits 리스트 뒤지기
for i, doc in enumerate(splits):
    if target_keyword in doc.page_content:
        found_docs.append((i, doc))

if found_docs:
    print(f"✅ 찾았다! 총 {len(found_docs)}개의 청크에 해당 금액이 있습니다.")
    print(f"   - 발견된 청크 인덱스: {[i for i, _ in found_docs]}")
    print("-" * 30)
    print(f"[내용 미리보기]\n{found_docs[0][1].page_content[:300]}...")
else:
    print("❌ 큰일났다... 데이터가 없다. 전처리 과정에서 삭제된 것 같다.")

print("\n" + "="*50 + "\n")

# 2. 검색 범위 늘려서 다시 테스트 (Retrieval Test)
query = "벤처확인종합관리시스템 기능 고도화 용역사업의 예산은 얼마인가요?"
print(f"🔍 범위 확장 검색 (k=10): {query}")

# k를 10으로 늘려서 순위권 밖의 문서를 낚아챔
results = vector_store.similarity_search(query, k=10)

found_rank = -1
for i, doc in enumerate(results):
    # 우리가 찾는 금액이 포함된 문서가 몇 등인지 확인
    if "352,000,000" in doc.page_content:
        found_rank = i + 1
        print(f"\n🎉 [Rank {found_rank}] 에서 정답 문서 발견!")
        print("-" * 30)
        print(doc.page_content[:500])
        break

if found_rank == -1:
    print("\n⚠️ 상위 10개 안에도 없습니다. (하지만 데이터가 있다면 LLM은 찾아낼 수 있습니다)")

🕵️‍♂️ '352,000,000' 데이터 수색 중...
✅ 찾았다! 총 1개의 청크에 해당 금액이 있습니다.
   - 발견된 청크 인덱스: [1]
------------------------------
[내용 미리보기]
별지서식 및 붙임 · 94

Ⅰ. 추진개요 1 추진배경 및 방향 □ 「벤처기업육성에 관한 특별조치법」(이하 ‘벤처기업법’) 복수의결권주식, 스톡옵 션(주식매수선택권), 성과조건부주식교부계약(RS) 등의 기능 고도화 및 이 관, 신규 구축업무를 本 과업에서 추진 ⚬ (복수의결권주식*) 벤처기업법 제16조의11에 따라 발행된 복수의결권 주식 보고 업무처리 시스템 구축 * 복수의결권이란? 모든 주는 1주당 하나의 의결권을 갖는 주평등원칙(상법)과 별도로 비상장 벤처기업 창업자에게만 1주당 최대 10배의 의결권 행사를 부여하는 제도 ⚬ (...


🔍 범위 확장 검색 (k=10): 벤처확인종합관리시스템 기능 고도화 용역사업의 예산은 얼마인가요?

🎉 [Rank 4] 에서 정답 문서 발견!
------------------------------
별지서식 및 붙임 · 94

Ⅰ. 추진개요 1 추진배경 및 방향 □ 「벤처기업육성에 관한 특별조치법」(이하 ‘벤처기업법’) 복수의결권주식, 스톡옵 션(주식매수선택권), 성과조건부주식교부계약(RS) 등의 기능 고도화 및 이 관, 신규 구축업무를 本 과업에서 추진 ⚬ (복수의결권주식*) 벤처기업법 제16조의11에 따라 발행된 복수의결권 주식 보고 업무처리 시스템 구축 * 복수의결권이란? 모든 주는 1주당 하나의 의결권을 갖는 주평등원칙(상법)과 별도로 비상장 벤처기업 창업자에게만 1주당 최대 10배의 의결권 행사를 부여하는 제도 ⚬ (스톡옵션*) 벤처기업법 제16조의3에 따라 부여된 벤처기업 스톡옵션 부여, 취소·철회 신고 및 업무시스템 구축 * 스톡옵션이란? 비상장 벤처기업 임·직원 및 기업 성장에 기여한 자에게 미리 정한 가격 (행사가격)으로 신주를 인수하거나 자기의 주식을 매수할 수 있는 권리 혹은 주식의 시